# Evaluating the Retrievers

## Load in documents and chunk them

In [1]:
from rag.documents import load_documents
from rag.chunk import chunk_documents

FILE_PATH = "/home/nick/github-projects/Sec-Rag/data/google_10K.pdf"

documents = load_documents(FILE_PATH)

chunks = chunk_documents(
    documents=documents,
    chunk_size=400,
    chunk_overlap=40
)

print(f"Length of Documents: {len(documents)}")
print(f"Length of Chunks: {len(chunks)}")

/home/nick/github-projects/Sec-Rag/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Length of Documents: 107
Length of Chunks: 1037


## Build every retriever

In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from rag.bm25 import BM25Retriever
from rag.dense import DenseRetriever
from rag.hybrid import HybridRetriever
from rag.hyde import HydeRetriever
from rag.llm import DEFAULT_CHAT_MODEL, LangChainGenerator
from rag.reranker import RerankerRetriever

load_dotenv()

bm25 = BM25Retriever()
bm25.add_documents(chunks)

dense = DenseRetriever()
dense.add_documents(chunks)

hybrid = HybridRetriever(bm25=bm25, dense=dense)

generator = LangChainGenerator(ChatOpenAI(model=DEFAULT_CHAT_MODEL, max_tokens=256))

retrievers = {
    "bm25": bm25,
    "dense": dense,
    "hybrid": hybrid,
    "reranker": RerankerRetriever(base_retriever=hybrid, candidate_pool=30),
    "hyde": HydeRetriever(base_retriever=dense, generator=generator),
}

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6917.19it/s]


## The questions and the pages that answer them

In [3]:
from rag.metrics import EvalExample

examples = [
    EvalExample("How much did Google spend on research and development in 2025?", {41, 55}),
    EvalExample("How much is Google paying to acquire Wiz?", {39, 46, 81}),
    EvalExample("What were Alphabet's total revenues in 2025?", {38, 39, 55, 66, 94}),
    EvalExample("How much did Google Cloud revenues grow from 2024 to 2025?", {38, 40}),
    EvalExample("What was the effective tax rate in 2025?", {43}),
    EvalExample("How much stock did Alphabet repurchase in 2025?", {45, 85}),
    EvalExample("How many employees did Alphabet have at the end of 2025?", {39}),
]

## Score each retriever

In [4]:
from rag.metrics import evaluate_retriever

results = {
    name: evaluate_retriever(retriever, examples, k=5)
    for name, retriever in retrievers.items()
}

metrics = list(results["bm25"].aggregate)

print(f"{'':10}" + "".join(f"{m:>13}" for m in metrics))
for name, result in results.items():
    print(f"{name:10}" + "".join(f"{result.aggregate[m]:>13.3f}" for m in metrics))

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2775.18it/s]


                  hit@5  precision@5     recall@5          mrr       ndcg@5          map
bm25              0.714        0.229        0.571        0.571        0.528        0.476
dense             0.714        0.171        0.433        0.571        0.439        0.383
hybrid            1.000        0.257        0.671        0.671        0.577        0.486
reranker          0.714        0.229        0.529        0.571        0.493        0.436
hyde              0.857        0.286        0.671        0.786        0.652        0.588


## Where each retriever misses

In [5]:
print(f"{'':62}" + "".join(f"{name:>10}" for name in results))
for i, example in enumerate(examples):
    hits = "".join(f"{results[name].per_example[i]['hit']:>10.0f}" for name in results)
    print(f"{example.question[:60]:62}{hits}")

                                                                    bm25     dense    hybrid  reranker      hyde
How much did Google spend on research and development in 202           1         1         1         1         1
How much is Google paying to acquire Wiz?                              1         1         1         1         1
What were Alphabet's total revenues in 2025?                           0         1         1         1         1
How much did Google Cloud revenues grow from 2024 to 2025?             1         1         1         1         1
What was the effective tax rate in 2025?                               1         1         1         1         1
How much stock did Alphabet repurchase in 2025?                        1         0         1         0         1
How many employees did Alphabet have at the end of 2025?               0         0         1         0         0
